# Personal AI Assistant — QLoRA Fine-Tuning Notebook

## 1. Environment Setup


In [1]:
# Core libraries for QLoRA fine-tuning
!pip install -q -U "transformers>=4.44.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
    "peft>=0.12.0" "bitsandbytes>=0.43.1" "trl>=0.9.6" "sentencepiece" \
    "evaluate" "scikit-learn" "pandas"

print("Installation complete.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 98.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 90.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 97.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.1 MB/s eta 0:00:00
   ━━━━

## 2. Imports

In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Total GPU memory (GB): 15.64


## 3. Configuration


In [3]:
class Config:
    # ---- Model ----
    model_name = "mistralai/Mistral-7B-Instruct-v0.2"   # 7B-parameter instruction-tuned model (at the size limit)
    model_supports_system_role = False    # Mistral's chat template has no "system" role (see Section 5)

    # ---- Data ----
    dataset_path = "/kaggle/input/datasets/rehabhamdy/personal-data/personal_finetuning_dataset.csv"   # path to the CSV dataset
    output_dir = "./personal-qlora-adapter"            # where adapter/checkpoints are saved
    val_split_ratio = 0.10                              # 90% train / 10% validation
    seed = 42

    # ---- Sequence length ----
    max_seq_length = 512   # reduce to 256 if you hit GPU memory limits

    # ---- 4-bit quantization (QLoRA) ----
    load_in_4bit = True
    bnb_4bit_quant_type = "nf4"          # NF4: normal-float 4-bit, best for LLM weights
    bnb_4bit_use_double_quant = True     # quantize the quantization constants too -> extra memory savings
    bnb_4bit_compute_dtype = "bfloat16"  # compute dtype during forward/backward (float16 if bf16 unsupported)

    # ---- LoRA ----
    lora_r = 16              # rank of the LoRA update matrices
    lora_alpha = 32           # scaling factor (commonly 2x r)
    lora_dropout = 0.05       # dropout inside LoRA layers, helps prevent overfitting
    lora_bias = "none"        # do not train bias terms
    lora_task_type = "CAUSAL_LM"
    # Mistral-7B is a LLaMA-style decoder architecture; these are its attention/MLP projection names
    lora_target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]

    # ---- Training ----
    num_train_epochs = 3             # small dataset -> few epochs to avoid overfitting
    learning_rate = 1e-4               # typical LoRA learning rate (higher than full fine-tuning)
    per_device_train_batch_size = 1    # kept small: 7B model leaves less headroom per step than a 1-2B model
    per_device_eval_batch_size = 1
    gradient_accumulation_steps = 16   # effective batch size = 1 * 16 = 16 (compensates for the small per-device batch)
    warmup_steps = 5                   # a handful of warmup steps; more stable than warmup_ratio on ~9 steps/epoch
    weight_decay = 0.01
    logging_steps = 1
    eval_strategy = "epoch"
    save_strategy = "epoch"
    save_total_limit = 2
    early_stopping_patience = 2        # stop if val loss doesn't improve for 2 evals
    gradient_checkpointing = True      # trade compute for memory
    bf16 = True                        # use bfloat16 mixed precision if supported
    fp16 = False                       # fallback if bf16 unsupported (set bf16=False, fp16=True)
    loss_type = "nll"                  # explicit standard loss (see Section 10 note on why this matters)

cfg = Config()
set_seed(cfg.seed)
random.seed(cfg.seed)
np.random.seed(cfg.seed)

# Auto-detect whether bf16 is actually supported on this GPU
if cfg.bf16 and torch.cuda.is_available() and not torch.cuda.is_bf16_supported():
    print("bf16 not supported on this GPU, falling back to fp16.")
    cfg.bf16 = False
    cfg.fp16 = True
    cfg.bnb_4bit_compute_dtype = "float16"

print("Configuration loaded.")


Configuration loaded.


## 4. Load the CSV Dataset

In [4]:
# If running in Colab and the file isn't already present, uncomment to upload it:
# from google.colab import files
# uploaded = files.upload()  # then select personal_finetuning_dataset.csv

df = pd.read_csv(cfg.dataset_path)

print(f"Number of examples: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Categories: {df['category'].nunique()} unique categories")
print()
print("Sample examples:")
df.sample(n=min(5, len(df)), random_state=cfg.seed)


Number of examples: 167
Columns: ['id', 'category', 'instruction', 'response']
Categories: 20 unique categories

Sample examples:


,id,category,instruction,response
101,102,Projects,Tell me about Rehab's Divorce Risk Prediction ...,Rehab built an LLM-guided feature engineering ...
108,109,Projects,Tell me about Rehab's Arabic Auto-Correction S...,Rehab developed a deep learning system for Ara...
78,79,LLMs,What experience does Rehab have with Large Lan...,Rehab has hands-on LLM experience including fi...
56,57,Machine Learning,What machine learning experience does Rehab have?,Rehab has broad machine learning experience sp...
98,99,Projects,What technologies were used to build the Multi...,The Multi-Org RAG Chatbot was built with Pytho...


In [5]:
# Quick sanity checks on the raw data
assert set(["id", "category", "instruction", "response"]).issubset(df.columns), \
    "Dataset must contain id, category, instruction, response columns"

print("Any missing values?")
print(df.isnull().sum())

print()
print("Category distribution:")
print(df["category"].value_counts())


Any missing values?
id             0
category       0
instruction    0
response       0
dtype: int64

Category distribution:
category
Projects                      41
General Questions About Me    32
Personal Background           10
Education                      8
LLMs                           8
Internships                    7
Courses                        6
Programming                    6
Technical Skills               6
Achievements                   6
Tools & Frameworks             5
Machine Learning               5
Deep Learning                  5
NLP                            5
Computer Vision                4
Work Experience                3
Certifications                 3
Generative AI                  3
Career Background              2
Interests                      2
Name: count, dtype: int64


## 5. Dataset Preprocessing

In [6]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = (
    "You are a personal AI assistant that answers questions about Rehab, "
    "based only on known facts about her background, education, skills, "
    "and projects."
)

def build_messages(question, answer=None):
    """Build a chat message list, respecting whether the model's template
    supports a dedicated system role (see Section 5 note)."""
    if cfg.model_supports_system_role:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ]
    else:
        # Fold the system instruction into the first user turn instead
        messages = [
            {"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{question}"},
        ]
    if answer is not None:
        messages.append({"role": "assistant", "content": answer})
    return messages


def format_example(row):
    """Build a single chat-formatted training string using the model's own
    chat template (do not hand-roll a generic format)."""
    messages = build_messages(row["instruction"], row["response"])
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return text

df["text"] = df.apply(format_example, axis=1)

print("Example formatted training text:\n")
print(df["text"].iloc[0])


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Example formatted training text:

<s> [INST] You are a personal AI assistant that answers questions about Rehab, based only on known facts about her background, education, skills, and projects.

Who is Rehab? [/INST] Rehab Hamdy Abdallah is an AI Engineer and Artificial Intelligence graduate with a strong foundation in machine learning, deep learning, Generative AI, and Large Language Models (LLMs). She has developed practical experience through academic and personal projects involving RAG systems, LLM fine-tuning, and AI-powered applications, and is passionate about learning and building practical AI solutions for real-world problems.</s>


## 6. Train/Validation Split (90% / 10%, reproducible)

In [7]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=cfg.val_split_ratio,
    random_state=cfg.seed,
    shuffle=True,
)

print(f"Train examples: {len(train_df)}")
print(f"Validation examples: {len(val_df)}")

train_dataset = Dataset.from_pandas(train_df[["text"]].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[["text"]].reset_index(drop=True))

dataset = DatasetDict({"train": train_dataset, "validation": val_dataset})
dataset


Train examples: 150
Validation examples: 17


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 150
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 17
    })
})

## 7. Load the Base Model

In [8]:
compute_dtype = getattr(torch, cfg.bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,          # NF4: normal-float 4-bit, tuned for weight distributions
    bnb_4bit_use_double_quant=cfg.bnb_4bit_use_double_quant,  # quantizes the quant constants -> saves more memory
    bnb_4bit_compute_dtype=compute_dtype,                  # dtype used for the actual matmuls
)

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(model.config)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal base model parameters: {n_params/1e9:.2f}B")


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

MistralConfig {
  "architectures": [
    "MistralForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 32768,
  "model_type": "mistral",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": true,
    "load_in_8bit": false,
    "quant_method": "bitsandbytes"
  },
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta

## 8. 4-bit Quantization Details

In [9]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=cfg.gradient_checkpointing,
)
print("Model prepared for k-bit (4-bit) training.")


Model prepared for k-bit (4-bit) training.


## 9. LoRA Configuration


In [10]:
lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias=cfg.lora_bias,
    task_type=cfg.lora_task_type,
    target_modules=cfg.lora_target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


## 10. Training

In [11]:
# --- Config addition ---
cfg.optim = "paged_adamw_8bit"   # QLoRA-standard, avoids the fused-optimizer device issue entirely
training_args = SFTConfig(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    warmup_steps=cfg.warmup_steps,
    weight_decay=cfg.weight_decay,
    logging_steps=cfg.logging_steps,
    eval_strategy=cfg.eval_strategy,
    save_strategy=cfg.save_strategy,
    save_total_limit=cfg.save_total_limit,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=cfg.bf16,
    fp16=cfg.fp16,
    optim=cfg.optim, 
    gradient_checkpointing=cfg.gradient_checkpointing,
    max_length=cfg.max_seq_length,
    dataset_text_field="text",
    packing=False,               # keep examples separate (they are short, distinct Q&A pairs)
    loss_type=cfg.loss_type,     # "nll" — avoids the chunked_nll/Accelerate-hook crash, see note above
    report_to="none",
    seed=cfg.seed,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg.early_stopping_patience)],
)

print("Trainer created successfully!")


Adding EOS to train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Trainer created successfully!


In [13]:
# --- Config addition ---
cfg.optim = "paged_adamw_8bit"   # QLoRA-standard, avoids the fused-optimizer device issue entirely
# --- Smoke test: use a throwaway trainer, don't reuse it afterward ---
smoke_args = training_args
smoke_args.max_steps = 3
smoke_trainer = SFTTrainer(
    model=model,
    args=smoke_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg.early_stopping_patience)],
)
smoke_result = smoke_trainer.train()
print(smoke_result)
del smoke_trainer   # discard — do not reuse its optimizer state
torch.cuda.empty_cache()

Adding EOS to train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
0,4.583558,3.972052,1.752054,0.440559,4556.000000


TrainOutput(global_step=3, training_loss=4.887419859568278, metrics={'train_runtime': 178.8682, 'train_samples_per_second': 0.268, 'train_steps_per_second': 0.017, 'total_flos': 195523559325696.0, 'train_loss': 4.887419859568278, 'epoch': 0.32})


In [14]:
# --- Full run: brand-new trainer, brand-new optimizer state ---
training_args.max_steps = -1   # ensure the real config isn't capped

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg.early_stopping_patience)],
)

train_result = trainer.train()
print(train_result)

Adding EOS to train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.466550,1.470850,1.375701,0.730895,13833.000000
2,0.951556,1.191294,0.908452,0.774004,27666.000000
3,0.771467,1.114439,0.820810,0.783875,41499.000000


TrainOutput(global_step=30, training_loss=1.461811222632726, metrics={'train_runtime': 1618.3674, 'train_samples_per_second': 0.278, 'train_steps_per_second': 0.019, 'total_flos': 1780955265245184.0, 'train_loss': 1.461811222632726, 'epoch': 3.0})


In [16]:
# Persist training metrics for later reference
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

with open(os.path.join(cfg.output_dir, "training_config.json"), "w") as f:
    json.dump({k: v for k, v in vars(Config).items() if not k.startswith("__")}, f, indent=2, default=str)

print("Training metrics and configuration saved to", cfg.output_dir)


***** train metrics *****
  epoch                    =        3.0
  total_flos               =  1658643GF
  train_loss               =     1.4618
  train_runtime            = 0:26:58.36
  train_samples_per_second =      0.278
  train_steps_per_second   =      0.019
Training metrics and configuration saved to ./personal-qlora-adapter


## 11. Evaluation

In [17]:
eval_metrics = trainer.evaluate()
print("Validation metrics:", eval_metrics)

val_loss = eval_metrics.get("eval_loss")
if val_loss is not None:
    perplexity = float(np.exp(val_loss))
    print(f"Validation perplexity: {perplexity:.2f}")

with open(os.path.join(cfg.output_dir, "eval_results.json"), "w") as f:
    json.dump(eval_metrics, f, indent=2, default=str)


Training Loss,Validation Loss,Epoch,Entropy,Mean Token Accuracy,Num Tokens
0.771467,1.114439,3,0.820810,0.783875,41499.000000


Validation metrics: {'eval_loss': 1.114438772201538, 'eval_entropy': 0.8208097219467163, 'eval_mean_token_accuracy': 0.7838748412973741, 'eval_num_tokens': 41499.0}
Validation perplexity: 3.05


In [18]:
def generate_response(model_to_use, question, max_new_tokens=150):
    """Generate an answer to `question` using the given model (base or fine-tuned)."""
    messages = build_messages(question)   # handles the no-system-role template correctly
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model_to_use.device)
    with torch.no_grad():
        output_ids = model_to_use.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Load a fresh, non-adapted copy of the base model for a fair "before" comparison
base_model_for_compare = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

test_questions = [
    "What AI experience does Rehab have?",
    "What did Rehab study at university?",
    "What is Rehab's graduation project about?",
    "What programming languages does Rehab know?",
    "Has Rehab worked with LoRA fine-tuning?",
]

comparison_results = []
for q in test_questions:
    base_answer = generate_response(base_model_for_compare, q)
    finetuned_answer = generate_response(model, q)
    comparison_results.append({
        "question": q,
        "base_model_answer": base_answer,
        "finetuned_model_answer": finetuned_answer,
    })
    print("=" * 80)
    print("Q:", q)
    print("-" * 80)
    print("FINE-TUNED MODEL:", finetuned_answer)

# Free the extra base-model copy
del base_model_for_compare
torch.cuda.empty_cache()

with open(os.path.join(cfg.output_dir, "base_vs_finetuned_comparison.json"), "w") as f:
    json.dump(comparison_results, f, indent=2)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Q: What AI experience does Rehab have?
--------------------------------------------------------------------------------
FINE-TUNED MODEL: Rehab has experience with AI and machine learning, including deep learning, neural networks, and generative AI, as well as experience with frameworks like TensorFlow, PyTorch, and Keras.
Q: What did Rehab study at university?
--------------------------------------------------------------------------------
FINE-TUNED MODEL: Rehab studied Artificial Intelligence at the Faculty of Computers and Artificial Intelligence, Helwan University, graduating in June 2027.
Q: What is Rehab's graduation project about?
--------------------------------------------------------------------------------
FINE-TUNED MODEL: Rehab's graduation project is a Multi-Org RAG Chatbot that integrates Qwen2.5-1.5B-Instruct with LoRA and PEFT to provide personalized knowledge base access, automated Q&A generation, and multi-organization RAG chatbot functionality.
Q: What programming 

In [19]:
def generate_response(model_to_use, question, max_new_tokens=150):
    """Generate an answer to `question` using the given model (base or fine-tuned)."""
    messages = build_messages(question)   # handles the no-system-role template correctly
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model_to_use.device)
    with torch.no_grad():
        output_ids = model_to_use.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Load a fresh, non-adapted copy of the base model for a fair "before" comparison
base_model_for_compare = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

test_questions = [
            "How did Rehab first become interested in artificial intelligence?",
        "What areas of technology has Rehab studied?",
        "Can you summarize Rehab's technical background?",
        "What is Rehab's experience with machine learning?",
        "What experience does Rehab have in natural language processing?",
        "What does Rehab know about deep learning?",
        "What experience does Rehab have with Generative AI?",
        "What kind of projects has Rehab worked on?",
        "What technologies has Rehab used in her AI projects?",
        "What is Rehab's background in Large Language Models?",
        "What fine-tuning techniques has Rehab learned?",
        "What are some of Rehab's main technical skills?",
        "How would you describe Rehab's journey into AI?",
        "What did Rehab work on during her graduation project?",
        "What programming skills has Rehab developed?",
        "Which areas of AI is Rehab most interested in?",
        "What professional experience does Rehab have in the AI field?",
        "Can you give me a brief professional profile of Rehab?",
        "How would you describe Rehab's educational background?",
        "What AI technologies is Rehab currently familiar with?",
]

comparison_results = []
for q in test_questions:
    base_answer = generate_response(base_model_for_compare, q)
    finetuned_answer = generate_response(model, q)
    comparison_results.append({
        "question": q,
        "base_model_answer": base_answer,
        "finetuned_model_answer": finetuned_answer,
    })
    print("=" * 80)
    print("Q:", q)
    print("-" * 80)
    print("FINE-TUNED MODEL:", finetuned_answer)

# Free the extra base-model copy
del base_model_for_compare
torch.cuda.empty_cache()

with open(os.path.join(cfg.output_dir, "base_vs_finetuned_comparison.json"), "w") as f:
    json.dump(comparison_results, f, indent=2)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Q: How did Rehab first become interested in artificial intelligence?
--------------------------------------------------------------------------------
FINE-TUNED MODEL: Rehab first became interested in artificial intelligence when she attended a Machine Learning Workshop in 2024, which sparked her curiosity and led her to pursue AI as a career.
Q: What areas of technology has Rehab studied?
--------------------------------------------------------------------------------
FINE-TUNED MODEL: Rehab has studied Computer Vision, Deep Learning, Generative AI, LLM Fine-Tuning, NLP, RAG Systems, and Transfer Learning.
Q: Can you summarize Rehab's technical background?
--------------------------------------------------------------------------------
FINE-TUNED MODEL: Yes, Rehab's technical background includes a B.Sc. in Artificial Intelligence (AI) from Helwan University, graduating in June 2027. Her skills include Python, C++, Java, MATLAB, TensorFlow, PyTorch, Keras, OpenCV, and Git. She has hand

## 12. Save the LoRA Adapter

In [20]:
adapter_dir = os.path.join(cfg.output_dir, "final_adapter")
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("LoRA adapter and tokenizer saved to:", adapter_dir)
print("Adapter files:")
for f in os.listdir(adapter_dir):
    path = os.path.join(adapter_dir, f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f}: {size_mb:.2f} MB")


LoRA adapter and tokenizer saved to: ./personal-qlora-adapter/final_adapter
Adapter files:
  tokenizer_config.json: 0.00 MB
  chat_template.jinja: 0.00 MB
  tokenizer.json: 3.51 MB
  adapter_config.json: 0.00 MB
  adapter_model.safetensors: 167.83 MB
  README.md: 0.01 MB


## 13. Optional: Merge the LoRA Adapter into the Base Model

In [21]:
RUN_MERGE = False  # set to True to actually perform the merge

if RUN_MERGE:
    # Reload the base model in fp16/bf16 (NOT 4-bit) for a clean merge
    merge_dtype = torch.bfloat16 if cfg.bf16 else torch.float16
    base_model_fp = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        torch_dtype=merge_dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    merged_model = PeftModel.from_pretrained(base_model_fp, adapter_dir)
    merged_model = merged_model.merge_and_unload()

    merged_dir = os.path.join(cfg.output_dir, "merged_model")
    merged_model.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)
    print("Merged model saved to:", merged_dir)
else:
    print("Skipping merge. Using the LoRA adapter on top of the 4-bit base model for inference instead.")


Skipping merge. Using the LoRA adapter on top of the 4-bit base model for inference instead.


In [25]:
import shutil
import os

# Path to your final adapter
adapter_dir = "./personal-qlora-adapter/final_adapter"

# Create ZIP file
zip_path = shutil.make_archive(
    "./personal-qlora-adapter/final_adapter",
    "zip",
    adapter_dir
)

print(f"ZIP file created: {zip_path}")
print(f"Size: {os.path.getsize(zip_path) / (1024**2):.2f} MB")

ZIP file created: /kaggle/working/personal-qlora-adapter/final_adapter.zip
Size: 74.16 MB


## 14. Inference Function


In [22]:
# Load the base model fresh + attach the saved adapter, for a clean inference setup
_inference_base = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
_inference_model = PeftModel.from_pretrained(_inference_base, adapter_dir)
_inference_model.eval()
_inference_tokenizer = AutoTokenizer.from_pretrained(adapter_dir)


def answer_about_me(question: str, max_new_tokens: int = 150) -> str:
    """Answer a personal question about Rehab using the fine-tuned model."""
    messages = build_messages(question)   # handles the no-system-role template correctly
    prompt = _inference_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _inference_tokenizer(prompt, return_tensors="pt").to(_inference_model.device)
    with torch.no_grad():
        output_ids = _inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.4,
            top_p=0.9,
            pad_token_id=_inference_tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return _inference_tokenizer.decode(generated, skip_special_tokens=True).strip()


# Example usage
print(answer_about_me("What projects has Rehab worked on?"))


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Rehab has worked on several projects, including GenT, Aspect-Based Sentiment Analysis, Auto-Correction System, Brain Tumor Segmentation, Arabic Auto-Correction System, and Multi-Org RAG Chatbot.


## 15. Interactive Chatbot


In [24]:
def chat_loop():
    print("Ask me anything about Rehab. Type 'exit' or 'quit' to stop.\n")
    while True:
        try:
            user_input = input("You: ")
        except EOFError:
            break
        if user_input.strip().lower() in {"exit", "quit"}:
            print("Assistant: Goodbye!")
            break
        response = answer_about_me(user_input)
        print(f"Assistant: {response}\n")

# Uncomment to run interactively (requires a live input prompt, e.g. in Colab/Jupyter):
# chat_loop()
